In [5]:
import pandas as pd
import numpy as np
import re
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pandas.errors import SettingWithCopyWarning
#from econml.dml import LinearDML
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

warnings.simplefilter("ignore", SettingWithCopyWarning)

## Load Datasets


In [6]:
# BoardGameGeek dataset
bgg_dataset = pd.read_csv("bgg_dataset.csv", sep=';')
bgg_dataset.head(5)

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
0,174430.0,Gloomhaven,2017.0,1,4,120,14,42055,"8,79",1,"3,86",68323.0,"Action Queue, Action Retrieval, Campaign / Bat...","Strategy Games, Thematic Games"
1,161936.0,Pandemic Legacy: Season 1,2015.0,2,4,60,13,41643,"8,61",2,"2,84",65294.0,"Action Points, Cooperative Game, Hand Manageme...","Strategy Games, Thematic Games"
2,224517.0,Brass: Birmingham,2018.0,2,4,120,14,19217,"8,66",3,"3,91",28785.0,"Hand Management, Income, Loans, Market, Networ...",Strategy Games
3,167791.0,Terraforming Mars,2016.0,1,5,120,12,64864,"8,43",4,"3,24",87099.0,"Card Drafting, Drafting, End Game Bonuses, Han...",Strategy Games
4,233078.0,Twilight Imperium: Fourth Edition,2017.0,3,6,480,14,13468,"8,70",5,"4,22",16831.0,"Action Drafting, Area Majority / Influence, Ar...","Strategy Games, Thematic Games"


In [7]:
# Product/edition dataset
products_dataset = pd.read_csv('products_with_edition.csv')
products_dataset.head(5)

,productid,objecttype,objectid,producthref,price,currency,currencystring,currencysymbol,condition,prettycondition,...,quantity,userId,username,user_country,user_city,version_name,version_href,game_name,game_href,linkeduserGeekMarket
0,3870857,thing,1,/market/product/3870857,30.00,USD,U.S. dollars,$,likenew,Like New,...,1,2954044,LeBao,United States,Tallahassee,Die Macher (Multilingual edition 2006),/boardgameversion/24534/multilingual-edition-2006,Die Macher,/boardgame/1/die-macher,NaN
1,3866054,thing,1,/market/product/3866054,180.00,USD,U.S. dollars,$,new,New,...,1,3677448,CarsonChandler,United States,Christiansburg,Die Macher (Spielworxx English/German edition),/boardgameversion/455802/spielworxx-englishger...,Die Macher,/boardgame/1/die-macher,NaN
2,3854476,thing,1,/market/product/3854476,41.99,USD,U.S. dollars,$,likenew,Like New,...,1,215794,Eric Engelmann,United States,Rockville,Die Macher (Spielworxx English/German edition),/boardgameversion/455802/spielworxx-englishger...,Die Macher,/boardgame/1/die-macher,NaN
3,3853310,thing,1,/market/product/3853310,60.00,EUR,Euros,€,new,New,...,1,1613268,JillySilly,Belgium,NaN,Die Macher (Spielworxx English/German edition),/boardgameversion/455802/spielworxx-englishger...,Die Macher,/boardgame/1/die-macher,NaN
4,3845442,thing,1,/market/product/3845442,50.00,USD,U.S. dollars,$,verygood,Very Good,...,1,38240,caesarmom,United States,Lorton,Die Macher (German-only second edition),/boardgameversion/24939/german-only-second-edi...,Die Macher,/boardgame/1/die-macher,NaN


## Currency conversion in product_dataset

In [10]:
def convert_to_usd(amount, currency, rates):
    currency = str(currency).upper()
    if currency not in rates or pd.isna(amount):
        return None
    return round(float(amount) * rates[currency], 2)

rates = {
    'USD': 1.0,
    'EUR': 1.10,
    'GBP': 1.30,
    'AUD': 0.65,
    'CAD': 0.75,
    'CHF': 1.05,
    'SGD': 0.73,
    'SEK': 0.10,
    'PLN': 0.24,
    'HUF': 0.0027,
    'MXN': 0.057,
    'NZD': 0.60,
    'JPY': 0.0073,
    'CNY': 0.14,
    'BRL': 0.19,
    'CZK': 0.042,
    'MYR': 0.21,
}

products_dataset['currency'] = products_dataset['currency'].str.upper()

products_dataset['price'] = products_dataset['price'] * products_dataset['currency'].map(rates)

products_dataset['currency'] = 'USD'

products_dataset['price'] = products_dataset['price'].round(2)
products_dataset.head(5)

,productid,objecttype,objectid,producthref,price,currency,currencystring,currencysymbol,condition,prettycondition,...,quantity,userId,username,user_country,user_city,version_name,version_href,game_name,game_href,linkeduserGeekMarket
0,3870857,thing,1,/market/product/3870857,30.00,USD,U.S. dollars,$,likenew,Like New,...,1,2954044,LeBao,United States,Tallahassee,Die Macher (Multilingual edition 2006),/boardgameversion/24534/multilingual-edition-2006,Die Macher,/boardgame/1/die-macher,NaN
1,3866054,thing,1,/market/product/3866054,180.00,USD,U.S. dollars,$,new,New,...,1,3677448,CarsonChandler,United States,Christiansburg,Die Macher (Spielworxx English/German edition),/boardgameversion/455802/spielworxx-englishger...,Die Macher,/boardgame/1/die-macher,NaN
2,3854476,thing,1,/market/product/3854476,41.99,USD,U.S. dollars,$,likenew,Like New,...,1,215794,Eric Engelmann,United States,Rockville,Die Macher (Spielworxx English/German edition),/boardgameversion/455802/spielworxx-englishger...,Die Macher,/boardgame/1/die-macher,NaN
3,3853310,thing,1,/market/product/3853310,66.00,USD,Euros,€,new,New,...,1,1613268,JillySilly,Belgium,NaN,Die Macher (Spielworxx English/German edition),/boardgameversion/455802/spielworxx-englishger...,Die Macher,/boardgame/1/die-macher,NaN
4,3845442,thing,1,/market/product/3845442,50.00,USD,U.S. dollars,$,verygood,Very Good,...,1,38240,caesarmom,United States,Lorton,Die Macher (German-only second edition),/boardgameversion/24939/german-only-second-edi...,Die Macher,/boardgame/1/die-macher,NaN


## Clean Product Dataset

In [12]:
cols_to_keep = ["objectid", "price", "version_name", "condition"]
products_clean = products_dataset.loc[:, cols_to_keep].copy()

# Drop missing IDs/prices
products_clean = products_clean.dropna(subset=["objectid", "price"])

# Convert types
products_clean["objectid"] = pd.to_numeric(products_clean["objectid"], errors="coerce").astype('Int64')
products_clean["price"] = pd.to_numeric(products_clean["price"], errors="coerce")

# Standardize condition
products_clean["condition"] = products_clean["condition"].astype(str).str.lower().str.strip()

# Keep only 'new' condition
products_clean = products_clean[products_clean["condition"] == "new"]

# Clean version names
products_clean["version_name"] = products_clean["version_name"].astype(str).str.strip()

# Remove duplicates
products_clean = products_clean.drop_duplicates(subset=["objectid", "version_name", "condition", "price"])

# Average price for duplicate versions
products_clean_avg = products_clean.groupby(
    ["objectid", "version_name", "condition"], as_index=False
)["price"].mean()

# IQR-based outlier removal
Q1 = products_clean_avg["price"].quantile(0.25)
Q3 = products_clean_avg["price"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

products_model_ready = products_clean_avg[
    (products_clean_avg["price"] >= lower_bound) & 
    (products_clean_avg["price"] <= upper_bound)
]

products_model_ready.head(5)

,objectid,version_name,condition,price
0,1,Die Macher (German-only second edition),new,73.63
1,1,Die Macher (Multilingual edition 2006),new,44.50
2,1,Die Macher (Spanish edition),new,46.20
3,1,Die Macher (Spielworxx English/German edition ...,new,71.50
8,5,Acquire (3M Bookshelf edition),new,64.50


## Clean BGG Dataset

In [13]:
# Numeric conversion
numeric_cols = ['Year Published', 'Rating Average', 'Complexity Average', 'BGG Rank']
int_cols = ['Min Players', 'Max Players', 'Play Time', 'Min Age', 'Users Rated', 'Owned Users']

for col in numeric_cols:
    bgg_dataset[col] = pd.to_numeric(bgg_dataset[col].astype(str).str.replace(',', '.'), errors='coerce')
for col in int_cols:
    bgg_dataset[col] = pd.to_numeric(bgg_dataset[col], errors='coerce')

# Strip text
for col in ['Name', 'Mechanics', 'Domains']:
    bgg_dataset[col] = bgg_dataset[col].astype(str).str.strip()

# Drop rows with missing values
bgg_cleaned = bgg_dataset.dropna().copy()

bgg_cleaned.head(5)

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
0,174430.0,Gloomhaven,2017.0,1,4,120,14,42055,8.79,1,3.86,68323.0,"Action Queue, Action Retrieval, Campaign / Bat...","Strategy Games, Thematic Games"
1,161936.0,Pandemic Legacy: Season 1,2015.0,2,4,60,13,41643,8.61,2,2.84,65294.0,"Action Points, Cooperative Game, Hand Manageme...","Strategy Games, Thematic Games"
2,224517.0,Brass: Birmingham,2018.0,2,4,120,14,19217,8.66,3,3.91,28785.0,"Hand Management, Income, Loans, Market, Networ...",Strategy Games
3,167791.0,Terraforming Mars,2016.0,1,5,120,12,64864,8.43,4,3.24,87099.0,"Card Drafting, Drafting, End Game Bonuses, Han...",Strategy Games
4,233078.0,Twilight Imperium: Fourth Edition,2017.0,3,6,480,14,13468,8.70,5,4.22,16831.0,"Action Drafting, Area Majority / Influence, Ar...","Strategy Games, Thematic Games"


## Merge Product and BGG datasets

In [14]:
bgg_cleaned['ID'] = pd.to_numeric(bgg_cleaned['ID'], errors='coerce')
products_model_ready['objectid'] = pd.to_numeric(products_model_ready['objectid'], errors='coerce')

# Filter products in BGG
products_model_ready = products_model_ready[products_model_ready['objectid'].isin(bgg_cleaned['ID'])]

# Aggregate by version
products_grouped = products_model_ready.groupby(['objectid', 'version_name'], as_index=False).agg({
    'price': 'mean',
    'condition': 'first'
})

# Merge
bgg_merged = bgg_cleaned.merge(
    products_grouped[['objectid', 'price', 'condition', 'version_name']],
    how='left', left_on='ID', right_on='objectid'
)
bgg_merged.drop(columns=['objectid'], inplace=True)

# Clean unrealistic values
bgg_merged = bgg_merged[
    (bgg_merged['Year Published'] >= 1900) &
    (bgg_merged['Min Players'] > 0) &
    (bgg_merged['Max Players'] <= 20) &
    (bgg_merged['Play Time'] <= 600) &
    (bgg_merged['Min Age'] >= 0) &
    ((bgg_merged['price'].isna()) | (bgg_merged['price'] <= 300))
]

bgg_final = bgg_merged.dropna(subset=['price', 'version_name', 'condition'])
bgg_final.rename(columns={'BGG Rank': 'Success Rank'}, inplace=True)

bgg_final.head(5)

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,Success Rank,Complexity Average,Owned Users,Mechanics,Domains,price,condition,version_name
0,174430.0,Gloomhaven,2017.0,1,4,120,14,42055,8.79,1,3.86,68323.0,"Action Queue, Action Retrieval, Campaign / Bat...","Strategy Games, Thematic Games",98.450000,new,"Gloomhaven (German edition, first printing)"
1,161936.0,Pandemic Legacy: Season 1,2015.0,2,4,60,13,41643,8.61,2,2.84,65294.0,"Action Points, Cooperative Game, Hand Manageme...","Strategy Games, Thematic Games",63.810000,new,Pandemic Legacy: Season 1 (English blue editio...
2,161936.0,Pandemic Legacy: Season 1,2015.0,2,4,60,13,41643,8.61,2,2.84,65294.0,"Action Points, Cooperative Game, Hand Manageme...","Strategy Games, Thematic Games",81.730000,new,Pandemic Legacy: Season 1 (English blue editio...
3,161936.0,Pandemic Legacy: Season 1,2015.0,2,4,60,13,41643,8.61,2,2.84,65294.0,"Action Points, Cooperative Game, Hand Manageme...","Strategy Games, Thematic Games",61.666667,new,Pandemic Legacy: Season 1 (English red edition...
4,161936.0,Pandemic Legacy: Season 1,2015.0,2,4,60,13,41643,8.61,2,2.84,65294.0,"Action Points, Cooperative Game, Hand Manageme...","Strategy Games, Thematic Games",77.833333,new,Pandemic Legacy: Season 1 (English red edition...


## Data Preprocessing and Labeling

In [17]:
# Complexity label
bgg_final["complexity_label"] = pd.cut(
    bgg_final["Complexity Average"], bins=[0, 2, 3.5, 5],
    labels=["Easy", "Medium", "Hard"], include_lowest=True
)

# High rating flag
bgg_final["highly_rated"] = bgg_final["Rating Average"] >= 8

# Game type classification
def classify_game_type(domains):
    if pd.isna(domains):
        return "Unknown"
    d = domains.lower()
    strategy_keywords = ["strategy", "abstract", "war", "wargame", "economic"]
    thematic_keywords = ["thematic", "adventure", "fantasy", "horror", "science fiction", "story", "exploration"]
    family_keywords = ["family", "children", "party", "puzzle", "dexterity", "trivia"]
    is_strategy = any(k in d for k in strategy_keywords)
    is_thematic = any(k in d for k in thematic_keywords)
    is_family = any(k in d for k in family_keywords)
    if is_strategy and is_thematic: return "Hybrid"
    elif is_strategy: return "Strategy"
    elif is_thematic: return "Thematic"
    elif is_family: return "Family/Party"
    else: return "Other"

bgg_final["game_type"] = bgg_final["Domains"].apply(classify_game_type)

# Price category
bgg_final["price_category"] = pd.cut(
    bgg_final["price"], bins=[0, 20, 50, 100, 500, 1000],
    labels=["very low", "low", "medium", "high", "very high"]
)

# Value-for-money features
bgg_final["rating_per_dollar"] = bgg_final["Rating Average"] / bgg_final["price"]
bgg_final["complexity_per_dollar"] = bgg_final["Complexity Average"] / bgg_final["price"]

# Mechanic annotation
mechanic_map = {
    "dice-driven": ["dice", "roll", "yahtzee"],
    "worker-placement": ["worker placement", "placement"],
    "deck-building": ["deck building", "deckbuilding"],
    "drafting": ["draft", "card drafting"],
    "tile-placement": ["tile placement", "domino", "polyomino"],
    "area-control": ["area control", "area majority"],
    "resource-management": ["resource", "economy", "produce"],
    "engine-building": ["engine", "combo", "synergy"],
    "set-collection": ["set collection"],
    "hand-management": ["hand management"],
    "card-based": ["card", "deck", "hand"],
    "co-op": ["cooperative", "co-op", "team"],
    "campaign-legacy": ["legacy", "campaign", "scenario"],
    "hidden-information": ["hidden", "secret", "deduction"],
    "bluffing": ["bluff", "traitor"],
    "puzzle": ["puzzle", "logic"],
    "party": ["party", "social"],
    "conflict-combat": ["combat", "fight", "battle"],
    "trading-negotiation": ["trade", "negotiation"]
}

def annotate_mechanics(text):
    if pd.isna(text):
        return ["other"]
    parts = [p.strip().lower() for p in text.split(",")]
    matched = set()
    for part in parts:
        for label, keywords in mechanic_map.items():
            if any(kw in part for kw in keywords):
                matched.add(label)
    return list(matched) if matched else ["other"]

bgg_final["Mechanics_annotated"] = bgg_final["Mechanics"].apply(annotate_mechanics)

# Explode for analysis
bgg_exploded = bgg_final.explode("Mechanics_annotated")

bgg_model_ready = bgg_final.drop(
    columns=["Name", "Mechanics", "Domains"]
)



,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,Success Rank,...,price,condition,version_name,complexity_label,highly_rated,game_type,price_category,rating_per_dollar,complexity_per_dollar,Mechanics_annotated
0,174430.0,Gloomhaven,2017.0,1,4,120,14,42055,8.79,1,...,98.450000,new,"Gloomhaven (German edition, first printing)",Hard,True,Hybrid,medium,0.089284,0.039208,"[conflict-combat, hand-management, card-based,..."
1,161936.0,Pandemic Legacy: Season 1,2015.0,2,4,60,13,41643,8.61,2,...,63.810000,new,Pandemic Legacy: Season 1 (English blue editio...,Medium,True,Hybrid,medium,0.134932,0.044507,"[set-collection, hand-management, card-based, ..."
2,161936.0,Pandemic Legacy: Season 1,2015.0,2,4,60,13,41643,8.61,2,...,81.730000,new,Pandemic Legacy: Season 1 (English blue editio...,Medium,True,Hybrid,medium,0.105347,0.034749,"[set-collection, hand-management, card-based, ..."
3,161936.0,Pandemic Legacy: Season 1,2015.0,2,4,60,13,41643,8.61,2,...,61.666667,new,Pandemic Legacy: Season 1 (English red edition...,Medium,True,Hybrid,medium,0.139622,0.046054,"[set-collection, hand-management, card-based, ..."
4,161936.0,Pandemic Legacy: Season 1,2015.0,2,4,60,13,41643,8.61,2,...,77.833333,new,Pandemic Legacy: Season 1 (English red edition...,Medium,True,Hybrid,medium,0.110621,0.036488,"[set-collection, hand-management, card-based, ..."


In [18]:
bgg_model_ready.head(5)


,ID,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,Success Rank,Complexity Average,...,price,condition,version_name,complexity_label,highly_rated,game_type,price_category,rating_per_dollar,complexity_per_dollar,Mechanics_annotated
0,174430.0,2017.0,1,4,120,14,42055,8.79,1,3.86,...,98.450000,new,"Gloomhaven (German edition, first printing)",Hard,True,Hybrid,medium,0.089284,0.039208,"[conflict-combat, hand-management, card-based,..."
1,161936.0,2015.0,2,4,60,13,41643,8.61,2,2.84,...,63.810000,new,Pandemic Legacy: Season 1 (English blue editio...,Medium,True,Hybrid,medium,0.134932,0.044507,"[set-collection, hand-management, card-based, ..."
2,161936.0,2015.0,2,4,60,13,41643,8.61,2,2.84,...,81.730000,new,Pandemic Legacy: Season 1 (English blue editio...,Medium,True,Hybrid,medium,0.105347,0.034749,"[set-collection, hand-management, card-based, ..."
3,161936.0,2015.0,2,4,60,13,41643,8.61,2,2.84,...,61.666667,new,Pandemic Legacy: Season 1 (English red edition...,Medium,True,Hybrid,medium,0.139622,0.046054,"[set-collection, hand-management, card-based, ..."
4,161936.0,2015.0,2,4,60,13,41643,8.61,2,2.84,...,77.833333,new,Pandemic Legacy: Season 1 (English red edition...,Medium,True,Hybrid,medium,0.110621,0.036488,"[set-collection, hand-management, card-based, ..."


In [19]:
bgg_exploded.head(5)


,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,Success Rank,...,price,condition,version_name,complexity_label,highly_rated,game_type,price_category,rating_per_dollar,complexity_per_dollar,Mechanics_annotated
0,174430.0,Gloomhaven,2017.0,1,4,120,14,42055,8.79,1,...,98.45,new,"Gloomhaven (German edition, first printing)",Hard,True,Hybrid,medium,0.089284,0.039208,conflict-combat
0,174430.0,Gloomhaven,2017.0,1,4,120,14,42055,8.79,1,...,98.45,new,"Gloomhaven (German edition, first printing)",Hard,True,Hybrid,medium,0.089284,0.039208,hand-management
0,174430.0,Gloomhaven,2017.0,1,4,120,14,42055,8.79,1,...,98.45,new,"Gloomhaven (German edition, first printing)",Hard,True,Hybrid,medium,0.089284,0.039208,card-based
0,174430.0,Gloomhaven,2017.0,1,4,120,14,42055,8.79,1,...,98.45,new,"Gloomhaven (German edition, first printing)",Hard,True,Hybrid,medium,0.089284,0.039208,co-op
0,174430.0,Gloomhaven,2017.0,1,4,120,14,42055,8.79,1,...,98.45,new,"Gloomhaven (German edition, first printing)",Hard,True,Hybrid,medium,0.089284,0.039208,campaign-legacy


In [20]:
bgg_final.head(5)

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,Success Rank,...,price,condition,version_name,complexity_label,highly_rated,game_type,price_category,rating_per_dollar,complexity_per_dollar,Mechanics_annotated
0,174430.0,Gloomhaven,2017.0,1,4,120,14,42055,8.79,1,...,98.450000,new,"Gloomhaven (German edition, first printing)",Hard,True,Hybrid,medium,0.089284,0.039208,"[conflict-combat, hand-management, card-based,..."
1,161936.0,Pandemic Legacy: Season 1,2015.0,2,4,60,13,41643,8.61,2,...,63.810000,new,Pandemic Legacy: Season 1 (English blue editio...,Medium,True,Hybrid,medium,0.134932,0.044507,"[set-collection, hand-management, card-based, ..."
2,161936.0,Pandemic Legacy: Season 1,2015.0,2,4,60,13,41643,8.61,2,...,81.730000,new,Pandemic Legacy: Season 1 (English blue editio...,Medium,True,Hybrid,medium,0.105347,0.034749,"[set-collection, hand-management, card-based, ..."
3,161936.0,Pandemic Legacy: Season 1,2015.0,2,4,60,13,41643,8.61,2,...,61.666667,new,Pandemic Legacy: Season 1 (English red edition...,Medium,True,Hybrid,medium,0.139622,0.046054,"[set-collection, hand-management, card-based, ..."
4,161936.0,Pandemic Legacy: Season 1,2015.0,2,4,60,13,41643,8.61,2,...,77.833333,new,Pandemic Legacy: Season 1 (English red edition...,Medium,True,Hybrid,medium,0.110621,0.036488,"[set-collection, hand-management, card-based, ..."


## Final datasets

In [22]:
bgg_final.to_csv("bgg_final.csv", index=False)
bgg_model_ready.to_csv("bgg_model_ready.csv", index=False)
bgg_exploded.to_csv("bgg_mechanics_exploded.csv", index=False)

print("All datasets saved successfully")
print("Model-ready dataset shape:", bgg_model_ready.shape)
print("Final BGG dataset shape:", bgg_final.shape)
print("Exploded mechanics dataset shape:", bgg_exploded.shape)

All datasets saved successfully
Model-ready dataset shape: (22826, 21)
Final BGG dataset shape: (22826, 24)
Exploded mechanics dataset shape: (49098, 24)
